# 04 · Similarity by physical characteristics

**Purpose.** Decide how to score the **physical** block: how well a marketed listing's building fits the
portfolio, for the brief's prompt 2 (a homogeneity score for each dimension, physical among them) and as one
input to prompt 1 (the most similar properties).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions).
The notebook ends with decisions to carry into `src/indurent_matching/features.py`.

**The marketed listings as supplied.** The raw file, before any cleaning, to show which fields there are to
score on. The physical fields are `Size sqm` (which holds square feet, EDA 2.1), `Min. Eaves m`, `Max. Eaves m`
and `Doors #`.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
pd.read_excel(ROOT / "data" / "Marketed Warehouses.xlsx", dtype=object)

,Property Id,Industrial Estate Name,Unit Name,Region,Latitude,Longitude,Car Parking Spaces #,Size sqm,Build Year,Yard Depth m,Min. Eaves m,Max. Eaves m,Doors #,EPC Rating
0,NaN,Cherry Lane,Unit 10,North West,52.467186,-3.784003,30,16860.130016,1977,NaN,NaN,NaN,NaN,NaN
1,NaN,Tech Hub,Unit 8,South East,NaN,NaN,23,905.523948,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Spitfire Park,Unit 42,Midlands,53.505831,-1.905054,49,2294.49811,1976,NaN,NaN,NaN,NaN,B
3,NaN,Stable Lane,Unit 14,South East,52.078733,-0.349448,48,617.297521,2000,NaN,7,7,16,C
4,NaN,Chancery Depot,Unit 2,NaN,52.485703,-4.478974,22,16613.786473,1966,NaN,5.21,5.46,3,A


* **Size is the only physical field every listing has.**
* **Eaves (min and max) and doors are blank for M1, M2 and M3**; only M4 (Stable Lane) and M5 (Chancery Depot)
  have them, and M4's 16 doors on 617 sq ft were flagged as implausible in cleaning.
* **Yard depth is blank for all five**, which is one reason it isn't used for similarity (EDA 2.7).
* So whatever the portfolio holds, a physical comparison for three of the five listings can only rest on size.

## 1. The question and the usable data

**Physical homogeneity** asks: *is this listing the kind of building Indurent owns?* The physical features are
size, eaves height (min and max) and doors. A value is usable only if it's recorded and not flagged doubtful;
doors also have to be the unit's own count, not an estate total copied onto every unit (EDA 2.6).

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")


def usable_physical(df):
    """The physical features with doubtful values, and door counts that aren't the unit's own, set to missing."""
    return pd.DataFrame({
        "size_sqft": df["size_sqft"].where(~df["size_sqft_doubtful"]),
        "min_eaves_m": df["min_eaves_m"].where(~df["min_eaves_m_doubtful"]),
        "max_eaves_m": df["max_eaves_m"].where(~df["max_eaves_m_doubtful"]),
        "doors": df["doors"].where(df["doors_describe_unit"] & ~df["doors_doubtful"]),
    })


physical = usable_physical(portfolio)
listing_physical = usable_physical(marketed)
pd.DataFrame({
    "portfolio units": physical.notna().sum(),
    "% of portfolio": (physical.notna().mean() * 100).round(0),
    "listings": listing_physical.notna().sum(),
})

,portfolio units,% of portfolio,listings
size_sqft,1240,99.0,5
min_eaves_m,409,33.0,2
max_eaves_m,270,22.0,2
doors,39,3.0,1


In [3]:
has = pd.DataFrame({
    "size": physical["size_sqft"].notna(),
    "eaves": physical[["min_eaves_m", "max_eaves_m"]].notna().any(axis=1),
    "doors": physical["doors"].notna(),
})
has.value_counts().rename("units").to_frame()

units
size  eaves doors       
True  False False    822
      True  False    379
            True      39
False False False      9
      True  False      1

In [4]:
listing_physical.round(1)

,size_sqft,min_eaves_m,max_eaves_m,doors
listing_id,,,,
M1,16860.1,NaN,NaN,NaN
M2,905.5,NaN,NaN,NaN
M3,2294.5,NaN,NaN,NaN
M4,617.3,7.0,7.0,NaN
M5,16613.8,5.2,5.5,3.0


* **Size is usable for almost every unit (1,240 of 1,250)**; the 10 without it are flagged under 500 sq ft.
* **Eaves for a third** (409 min, 270 max); **doors for only 39 units (3%)**, once estate totals and doubtful
  counts are left out.
* Most units can only be compared on size: **822 have size alone**, 379 size and eaves, 39 all three.
* The listings are the same: **M1, M2 and M3 have size only**; M4 has size and eaves (7 m; its 16 doors are
  flagged doubtful); M5 has size, eaves (5.2-5.5 m) and 3 doors.
* So **the physical block is mostly about size**, with eaves as a possible second feature. Doors are too sparse
  to matter: a listing's doors could be compared with only 39 portfolio units.

**What goes into the physical score**

| Feature | Decision | Why |
|---|---|---|
| Size | Scored, on a log scale | Usable for every listing and 99% of the portfolio |
| Eaves height | Context only, not scored (decided in section 4): the average of min and max, or whichever one is recorded | Clear height is what occupiers care about beyond floor area (how high they can rack), but only two listings have it. Using either side gives 419 units (260 both, 149 min only, 10 max only); where both exist they're close (median gap 1.1 m) |
| Doors | Context only, not scored | Only 39 portfolio units have a usable count of their own, and only M5 among the listings; counts rise with size (rank correlation 0.86), so they'd mostly repeat it |
| Volume (size x eaves) | Context only, not scored | 99% correlated with size (84% of its variation is size), needs eaves (419 units, two listings), and treats a big low shed and a small tall one as the same. Size and eaves scored separately capture both parts of it |

## 2. Size

In [5]:
size = physical["size_sqft"].dropna()
ticks = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000, 1_000_000]

fig = px.histogram(x=np.log10(size), nbins=50, height=400,
                   title="Portfolio unit sizes, log scale (dashed lines: listings)",
                   labels={"x": "Size (sq ft, log scale)"})
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
fig.update_layout(yaxis_title="Units")
for sqft in listing_physical["size_sqft"].dropna():
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick")
fig.show()

In [6]:
print(f"Portfolio size: median {size.median():,.0f} sq ft; 10th-90th percentile "
      f"{size.quantile(0.1):,.0f}-{size.quantile(0.9):,.0f}; range {size.min():,.0f}-{size.max():,.0f}")
listing_size = listing_physical[["size_sqft"]].round(0)
listing_size["% of portfolio units smaller"] = [round((size < s).mean() * 100) for s in listing_size["size_sqft"]]
listing_size

Portfolio size: median 4,753 sq ft; 10th-90th percentile 1,235-44,393; range 505-1,195,451


,size_sqft,% of portfolio units smaller
listing_id,,
M1,16860.0,79
M2,906.0,5
M3,2294.0,26
M4,617.0,1
M5,16614.0,78


* Size is **very skewed**, so it's shown (and should be compared) on a **log scale**: a 2,000 vs 4,000 sq ft
  difference should count the same as 20,000 vs 40,000. On that scale the portfolio has one broad peak around
  2,000-8,000 sq ft (median 4,753) and a long tail of big sheds up to 1.2m sq ft.
* The listings fall into two groups:
  * **small: M4 (617 sq ft), M2 (906) and M3 (2,294)**. M4 and M2 are smaller than 99% and 95% of portfolio
    units; M3 is in the lower-middle;
  * **mid-sized: M1 (16,860) and M5 (16,614)**, both larger than about 80% of portfolio units.
* None sits in the busiest part of the distribution, so the size score will separate them. Section 3 builds it.

## 3. A size score

The same idea as the age score: count portfolio units in **size bands**, and divide by the count in the busiest
band, so the most common size scores 1. Because size is compared on a log scale, the bands **double** in width
(500-1k, 1k-2k, 2k-4k sq ft, ...), so each band is the same width on the log scale.

In [7]:
edges = 500 * 2.0 ** np.arange(0, 13)  # 500, 1,000, 2,000, ... 2,048,000 sq ft


def short(sqft):
    """Readable band edge: 500, 1k, 2k, ... 512k, 1m, 2m."""
    if sqft < 1_000:
        return f"{sqft:.0f}"
    return f"{sqft / 1e6:.0f}m" if sqft >= 1e6 else f"{sqft / 1e3:.0f}k"


band_labels = [f"{short(lo)}-{short(hi)}" for lo, hi in zip(edges[:-1], edges[1:])]
size_band = pd.cut(size, edges, right=False, labels=band_labels)
per_band = size_band.value_counts(sort=False)
band_score = per_band / per_band.max()

fig = px.bar(x=band_labels, y=band_score.values, height=380,
             title="Score per size band: units in the band / units in the busiest band",
             labels={"x": "Size band (sq ft)", "y": "Score"})
fig.update_xaxes(type="category")
fig.show()
pd.DataFrame({"units": per_band, "score": band_score.round(2)}).T

size_sqft,500-1k,1k-2k,2k-4k,4k-8k,8k-16k,16k-32k,32k-64k,64k-128k,128k-256k,256k-512k,512k-1m,1m-2m
units,75.00,205.00,265.0,252.00,166.00,105.0,81.00,39.00,27.0,17.00,7.00,1.0
score,0.28,0.77,1.0,0.95,0.63,0.4,0.31,0.15,0.1,0.06,0.03,0.0


In [8]:
listing_size["band"] = pd.cut(listing_size["size_sqft"], edges, right=False, labels=band_labels)
listing_size["size score"] = listing_size["band"].map(band_score).astype(float).round(2)
listing_size

,size_sqft,% of portfolio units smaller,band,size score
listing_id,,,,
M1,16860.0,79,16k-32k,0.40
M2,906.0,5,500-1k,0.28
M3,2294.0,26,2k-4k,1.00
M4,617.0,1,500-1k,0.28
M5,16614.0,78,16k-32k,0.40


* The busiest band is **2k-4k sq ft** (265 units), so it scores 1; 1k-8k sq ft all score 0.77 or more, and
  the score falls away for big sheds (0.10 or less above 128k).
* The listings: **M3 1.00** (in the busiest band); **M1 and M5 0.40**; **M2 and M4 0.28**.
* **A boundary problem, unlike age.** M1 (16,860) and M5 (16,614) sit just above the 16k edge: at 15,900 sq ft
  they'd score 0.63, not 0.40. For age the jump at 1970 was in the data; here the distribution slopes smoothly,
  so the jump is made by the bands. Bands this wide (each a doubling) also lump M2 (906) and M4 (617) together,
  though M4 is far rarer: only 1% of portfolio units are smaller than it, against 5% for M2.

**Alternative: a moving band centred on the listing.** Count portfolio units within a doubling-wide window
*centred on the listing's own size* (from size / 1.41 to size x 1.41), and divide by the largest such count
anywhere in the portfolio. Same idea and same width, but no fixed edges to fall either side of.

In [9]:
sorted_size = np.sort(size.to_numpy())


def units_in_window(sqft):
    """Portfolio units within a doubling-wide window centred on sqft (sqft / sqrt 2 to sqft x sqrt 2)."""
    return np.searchsorted(sorted_size, sqft * np.sqrt(2)) - np.searchsorted(sorted_size, sqft / np.sqrt(2))


grid = np.geomspace(size.min(), size.max(), 2000)
window_counts = np.array([units_in_window(g) for g in grid])
busiest = window_counts.max()

fig = px.line(x=np.log10(grid), y=window_counts / busiest, height=380,
              title="Moving-window size score (dashed lines: listings)",
              labels={"x": "Size (sq ft, log scale)", "y": "Score"})
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
for sqft in listing_size["size_sqft"]:
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick")
fig.show()

print(f"Busiest window: {busiest} units, centred on about {grid[window_counts.argmax()]:,.0f} sq ft")
listing_size["moving-window score"] = [round(units_in_window(s) / busiest, 2) for s in listing_size["size_sqft"]]
listing_size[["size_sqft", "band", "size score", "moving-window score"]]

Busiest window: 297 units, centred on about 4,197 sq ft


,size_sqft,band,size score,moving-window score
listing_id,,,,
M1,16860.0,16k-32k,0.40,0.41
M2,906.0,500-1k,0.28,0.38
M3,2294.0,2k-4k,1.00,0.88
M4,617.0,500-1k,0.28,0.18
M5,16614.0,16k-32k,0.40,0.41


* The moving-window curve is **smooth**: no jumps, so a listing's score doesn't depend on which side of an edge
  it falls. It peaks around 4,200 sq ft, the middle of the portfolio's broad peak.
* **M1 and M5 (0.41)** score about the same as with bands, but now for the right reason: their score wouldn't
  change at 15,900 sq ft.
* **M2 (0.38) and M4 (0.18) are now told apart**: M4, smaller than 99% of the portfolio, scores half of M2.
* **M3 (0.88)** is just off the peak, rather than sitting in the busiest band at exactly 1.
* For age, fixed bands were fine because the distribution was flat and its one jump (1970) was real. Size
  slopes steadily, so fixed bands distort it more; the moving window is the better fit here.

## 4. Size and eaves height together

With doors and volume left out, the physical block has **two dimensions: size and eaves height**. Eaves height
is the average of min and max where both are recorded, or whichever one is. The scatter shows the 418 portfolio
units with both, and the listings: **M4 and M5** have both (stars); **M1, M2 and M3** have size only, so they're
shown as dashed lines at their size.

In [10]:
import plotly.graph_objects as go


def eaves_height(df):
    """Average of min and max eaves where both are recorded, otherwise whichever one is."""
    return df[["min_eaves_m", "max_eaves_m"]].mean(axis=1)


both = physical.assign(eaves_m=eaves_height(physical)).dropna(subset=["size_sqft", "eaves_m"])
both = both.join(portfolio[["industrial_estate_name", "unit_name"]])
listings_2d = listing_physical.assign(eaves_m=eaves_height(listing_physical))

fig = px.scatter(both, x=np.log10(both["size_sqft"]), y="eaves_m", hover_name="industrial_estate_name",
                 hover_data={"unit_name": True, "size_sqft": ":,.0f"}, opacity=0.5, height=480,
                 title="Portfolio units with size and eaves height, and the listings",
                 labels={"x": "Size (sq ft, log scale)", "eaves_m": "Eaves height (m)"})
fig.update_traces(marker={"size": 6}, name="portfolio unit", showlegend=True)
with_eaves = listings_2d.dropna(subset=["eaves_m"])
fig.add_trace(go.Scatter(x=np.log10(with_eaves["size_sqft"]), y=with_eaves["eaves_m"], mode="markers+text",
                         text=with_eaves.index, textposition="top center", name="listing (size and eaves)",
                         marker={"symbol": "star", "size": 16, "color": "firebrick"}))
for sqft in listings_2d.loc[listings_2d["eaves_m"].isna(), "size_sqft"]:
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick", opacity=0.5)
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
fig.show()

print(f"{len(both)} portfolio units with size and eaves; "
      f"rank correlation of eaves with size: {both['eaves_m'].corr(both['size_sqft'], method='spearman'):.2f}")
for listing_id, row in with_eaves.iterrows():
    peers = both[(both["size_sqft"] > row["size_sqft"] / 1.41) & (both["size_sqft"] < row["size_sqft"] * 1.41)]
    print(f"{listing_id}: {row['size_sqft']:,.0f} sq ft, {row['eaves_m']:.1f} m eaves; {len(peers)} portfolio units "
          f"of similar size have eaves, median {peers['eaves_m'].median():.1f} m "
          f"(10th-90th percentile {peers['eaves_m'].quantile(0.1):.1f}-{peers['eaves_m'].quantile(0.9):.1f} m)")

418 portfolio units with size and eaves; rank correlation of eaves with size: 0.57
M4: 617 sq ft, 7.0 m eaves; 6 portfolio units of similar size have eaves, median 4.5 m (10th-90th percentile 4.3-6.5 m)
M5: 16,614 sq ft, 5.3 m eaves; 45 portfolio units of similar size have eaves, median 6.3 m (10th-90th percentile 5.1-8.6 m)


* **Eaves rise with size** (rank correlation 0.57), but with a wide spread: units of the same size can differ
  by several metres. So eaves add real information beyond size, which is why it's worth keeping as the second
  dimension.
* **M4 is tall for its size**: 7 m eaves on 617 sq ft, above all but a few of the 6 portfolio units of similar
  size with eaves recorded (median 4.5 m, 90th percentile 6.5 m). With only 6 comparable units, though, that
  judgement rests on very little data. (M4's 16 doors were already flagged as implausible for its size.)
* **M5 is a little low for its size**: 5.3 m against a median of 6.3 m among 45 similar-sized units, but well
  within their range (5.1-8.6 m).
* **M1, M2 and M3** (dashed lines) can only be placed on size.
* **Decision: score on size only, and report eaves as context.** Only M4 and M5 have eaves, so a size-and-eaves
  score would put them on a different basis from M1, M2 and M3. Scoring size alone keeps all five comparable;
  eaves still appear in answers, e.g. *"M4's eaves (7 m) are tall for a unit of its size."*

## 5. Decisions for `features.py`

| Topic | Decision | Why |
|---|---|---|
| Physical score | **Size only**, scored with a moving window: portfolio units within a doubling-wide window centred on the listing's size (size / 1.41 to size x 1.41), divided by the busiest such window in the portfolio (0-1) | Size is the only physical field every listing has, so all five are scored on the same basis. The moving window has no band edges: fixed bands made a jump at 16k sq ft that isn't in the data, and lumped M2 with the much rarer M4 |
| Scale | Log (each window spans a doubling) | Size runs from 505 to 1.2m sq ft; a 2k vs 4k difference should count the same as 20k vs 40k |
| Doubtful sizes | Left out of the counts | The 10 units under 500 sq ft (EDA 2.1) |
| Eaves height | Context only: average of min and max, or whichever is recorded, compared with similar-sized portfolio units | Only M4 and M5 have it; scoring it would put them on a different basis. Still useful in answers ("tall for its size") |
| Doors | Context only | 39 usable portfolio counts, one usable listing count; they rise with size anyway |
| Volume | Context only | 99% correlated with size, needs eaves, and treats a big low shed and a small tall one as the same |
| Not used | Fixed size bands | Edge jumps made by the bands, and too coarse at the small end |